In [2]:
from als_model import RecommendedSystem
from repository_my_sql import RepositoryMySql
from data_manager import DataManager
from predictor import Predictor
import torch
from utils import collate
import numpy as np
import pandas as pd

def take_proteins_unique(dataframe):
    proteins = dataframe[['protein_id','protein_structure']].drop_duplicates().values.tolist()
    return proteins 

repo = RepositoryMySql()
# data = RepositoryMySql().read_interactions_for_als()
predictor = Predictor()

# print(data.count)
model = RecommendedSystem()
data_manager = DataManager()

model.load_pipeline()
df = model.calculate_recommended_proteins(5)
ids = df.toPandas().values.tolist()
interactions_with_structures =  repo.take_structures(ids)
proteins_to_analize = take_proteins_unique(interactions_with_structures)
data_manager.protein_graph_construct(proteins_to_analize)
dataset = data_manager.prepare_set(interactions_with_structures.values.tolist())
predictor.load_model("GAT")
test_loader = torch.utils.data.DataLoader(dataset, batch_size=len(dataset), shuffle=False, collate_fn=collate)

prediction = predictor.make_prediction(test_loader)

/Users/mircoperna/Desktop/ALS/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/28 14:50:12 WARN Utils: Your hostname, MacBook-Pro-di-Mirco.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.59 instead (on interface en0)
26/06/28 14:50:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 14:50:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/28 14:50:13 WARN Utils: Service 'SparkUI

loading tensors ...


  0%|          | 0/2068 [00:00<?, ?it/s]/Users/mircoperna/Desktop/ALS/utils.py:85: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  GCNData_mol = DATA.Data(x=torch.Tensor(mol_features),
100%|██████████| 2068/2068 [00:01<00:00, 2028.46it/s]


CPI_GAT_NFM_GLOBAL loading ...
Make prediction for 2068 samples...


In [3]:
print(prediction)

                                         drug_structure protein_structure  \
0                       C1CN(CC2=C1SC=C2)CC3=CC=CC=C3Cl            P25787   
1     C1CN(CCC12C(=O)NCN2C3=CC=CC=C3)CCCC(C4=CC=C(C=...            P25787   
2                    CN(C)CCCN1C2=CC=CC=C2SC3=CC=CC=C31            P25787   
3                           C[C@H](C(C)(C)C)O[P@H](=O)C            P25787   
4              CNCCC(C1=CC=CC=C1)OC2=CC=C(C=C2)C(F)(F)F            P25787   
...                                                 ...               ...   
2063              COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N            P10253   
2064  CN(CC1=C(C=CC(=C1)OC)OC)C2=CN=C3C(=C2)C(=NC(=N...            P10253   
2065  CC1=C2C(=NC(=NC2=NC=C1/C=CC3=C(C=CC(=C3)OC)OC)N)N            P10253   
2066  C1C=C[C@H]2C(=C1)C=CC3=C(N2CC4=CN=C5C(=N4)C(=N...            P10253   
2067                              CC(CC(=O)O)(CC(=O)O)O            P28070   

      prediction  
0       0.999514  
1       0.999813  
2       0.993747  

In [13]:
interactions_with_structures['drug_structure'] = interactions_with_structures['drug_structure'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
interactions_with_structures['protein_structure'] = interactions_with_structures['protein_structure'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
print(interactions_with_structures)

      drug_id                                     drug_structure protein_id  \
0        1479                    C1CN(CC2=C1SC=C2)CC3=CC=CC=C3Cl     P25787   
1        1112  C1CN(CCC12C(=O)NCN2C3=CC=CC=C3)CCCC(C4=CC=C(C=...     P25787   
2         987                 CN(C)CCCN1C2=CC=CC=C2SC3=CC=CC=C31     P25787   
3         459                        C[C@H](C(C)(C)C)O[P@H](=O)C     P25787   
4         403           CNCCC(C1=CC=CC=C1)OC2=CC=C(C=C2)C(F)(F)F     P25787   
...       ...                                                ...        ...   
2063     2667              COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N     P10253   
2064     2621  CN(CC1=C(C=CC(=C1)OC)OC)C2=CN=C3C(=C2)C(=NC(=N...     P10253   
2065     2545  CC1=C2C(=NC(=NC2=NC=C1/C=CC3=C(C=CC(=C3)OC)OC)N)N     P10253   
2066     2482  C1C=C[C@H]2C(=C1)C=CC3=C(N2CC4=CN=C5C(=N4)C(=N...     P10253   
2067     2469                              CC(CC(=O)O)(CC(=O)O)O     P28070   

                                      protein_struc

In [18]:
data_frame = pd.merge(interactions_with_structures, prediction,
                        on='drug_structure',
                        how='left')
data_frame = data_frame[['drug_id', 'protein_id','prediction', 'drug_structure', 'protein_structure_x']]
print(data_frame)

      drug_id protein_id  prediction  \
0        1479     P25787    0.999514   
1        1479     P25787    1.000000   
2        1479     P25787    1.000000   
3        1479     P25787    0.999944   
4        1112     P25787    0.999813   
...       ...        ...         ...   
5407     2545     P10253    1.000000   
5408     2482     P10253    1.000000   
5409     2469     P28070    0.008743   
5410     2469     P28070    0.510456   
5411     2469     P28070    0.007697   

                                         drug_structure  \
0                       C1CN(CC2=C1SC=C2)CC3=CC=CC=C3Cl   
1                       C1CN(CC2=C1SC=C2)CC3=CC=CC=C3Cl   
2                       C1CN(CC2=C1SC=C2)CC3=CC=CC=C3Cl   
3                       C1CN(CC2=C1SC=C2)CC3=CC=CC=C3Cl   
4     C1CN(CCC12C(=O)NCN2C3=CC=CC=C3)CCCC(C4=CC=C(C=...   
...                                                 ...   
5407  CC1=C2C(=NC(=NC2=NC=C1/C=CC3=C(C=CC(=C3)OC)OC)N)N   
5408  C1C=C[C@H]2C(=C1)C=CC3=C(N2CC4=CN=C5C(=N4

In [19]:
data_frame.to_csv('results.csv', index=False)